In [1]:
from typing import Any

from kirin.ir import Method
import matplotlib.pyplot as plt
import numpy as np

from bloqade import squin, tsim
from bloqade.cirq_utils import emit_circuit, load_circuit, noise
from bloqade.pyqrack import StackMemorySimulator
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList
import matplotlib.pyplot as plt

# this will help us have return types for our methods that have more intuitive names
Register = IList[Qubit, Any]
Measurement = IList[MeasurementResult, Any]

# this function will help us visualize some circuits
def show_circuit(squin_kernel):
    @squin.kernel
    def _to_visualize():
        _ = squin_kernel()

    return tsim.Circuit(_to_visualize).diagram(height=400)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [2]:
@squin.kernel
def steane_code_encoding(q):

    squin.h(q[4])
    squin.h(q[5])
    squin.h(q[6])

    squin.cx(q[0], q[1])
    squin.cx(q[0], q[2])

    squin.cx(q[6], q[0])
    squin.cx(q[6], q[1])
    squin.cx(q[6], q[3])

    squin.cx(q[5], q[0])
    squin.cx(q[5], q[2])
    squin.cx(q[5], q[3])

    squin.cx(q[4], q[3])
    squin.cx(q[4], q[2])
    squin.cx(q[4], q[1])

    return q

@squin.kernel
def steane_code_encoding_print():
    q = squin.qalloc(7)
    q = steane_code_encoding(q)
    return q

show_circuit(steane_code_encoding_print)

In [3]:
@squin.kernel
def logical_H(q):
    squin.broadcast.h(q)
    return q

@squin.kernel
def logical_S(q):
    squin.broadcast.s(q)
    return q

def logical_CX(control, target):
    for i in range(7):
        squin.cx(control[i], target[i])

@squin.kernel
def logical_T_injection(q):
    squin.h(q[0])
    squin.t(q[0])
    q = steane_code_encoding(q)
    return q

@squin.kernel
def logical_T_injection_print():
    q = squin.qalloc(7)
    q = logical_T_injection(q)
    return q

show_circuit(logical_T_injection_print)


In [4]:
@squin.kernel
def logical_T_teleportation(q):
    ancilla = squin.qalloc(7)
    ancilla = logical_T_injection(ancilla)
    
    for i in range(7):
        squin.cx(q[i], ancilla[i])

    bits = squin.broadcast.measure(ancilla)

    res = 0
    for b in bits:
        res ^= b

    if res == 1:
        q = logical_S(q)
    
    return q

In [5]:
@squin.kernel
def reference_t_gate(theta: float, phi: float):
    q = squin.qalloc(7)
    
    # 1. Arbitrary State Prep
    squin.ry(theta, q[0])
    squin.rz(phi, q[0])
    
    # 2. Physical T-Gate
    squin.t(q[0])
    
    q = steane_code_encoding(q)
    return q

In [6]:
@squin.kernel
def logical_t_gadget(theta: float, phi: float):
    q = squin.qalloc(7)
    
    # 1. Arbitrary State Prep on Data[0]
    squin.ry(theta, q[0])
    squin.rz(phi, q[0])
    steane_code_encoding(q)

    q = logical_T_teleportation(q)
        
    return q

In [7]:
theta = np.pi/4
phi = np.pi/3

reference_sim = StackMemorySimulator(min_qubits=7)
reference_task = reference_sim.task(reference_t_gate, args=(theta, phi))
reference_result = reference_task.run()
rho_reference = StackMemorySimulator.reduced_density_matrix(reference_result)
print(rho_reference)

logical_sim = StackMemorySimulator(min_qubits=7)
logical_task = logical_sim.task(logical_t_gadget, args=(theta, phi))
logical_result = logical_task.run()
rho_logical = StackMemorySimulator.reduced_density_matrix(logical_result)
print(rho_logical)

print("Do the two states match?", np.allclose(rho_reference, rho_logical))

[[ 0.10669413+0.j         0.        +0.j         0.        +0.j
  ...  0.        +0.j         0.        +0.j
  -0.0114383 -0.0426883j]
 [ 0.        +0.j         0.        +0.j         0.        +0.j
  ...  0.        +0.j         0.        +0.j
   0.        +0.j       ]
 [ 0.        +0.j         0.        +0.j         0.        +0.j
  ...  0.        +0.j         0.        +0.j
   0.        +0.j       ]
 ...
 [ 0.        +0.j         0.        +0.j         0.        +0.j
  ...  0.        +0.j         0.        +0.j
   0.        +0.j       ]
 [ 0.        +0.j         0.        +0.j         0.        +0.j
  ...  0.        +0.j         0.        +0.j
   0.        +0.j       ]
 [-0.0114383 +0.0426883j  0.        +0.j         0.        +0.j
  ...  0.        +0.j         0.        +0.j
   0.01830584+0.j       ]]
[[0.10669418+0.j         0.        +0.j         0.        +0.j
  ... 0.        +0.j         0.        +0.j
  0.01143829+0.04268831j]
 [0.        +0.j         0.        +0.j         0. 

In [8]:
import json

gridsynth_results = {}

with open("gridsynth_results.json", "r") as f:
    loaded_data = json.load(f)
    for k, v in loaded_data.items():
        gridsynth_results[eval(k)] = v

print("Loaded Gridsynth results:")
for key, value in gridsynth_results.items():
    print(f"n={key[0]}, eps={key[1]}: {value}")

Loaded Gridsynth results:
n=3, eps=0.01: HTHTSHTHTSHTSHTHTHTSHTSHTHTHTSHTSHTSHTHTSHTHTSHSSW
n=3, eps=0.001: HTSHTSHTSHTHTHTHTHTHTHTSHTHTHTSHTSHTHTSHTSHTHTSHTSHTSHTSHTHTSHTSHTSHTHTHTHTHTSHTHTHTHXSSWWWWW
n=3, eps=0.0001: HTHTHTSHTSHTHTHTHTHTSHTSHTHTHTHTHTSHTHTHTHTSHTSHTHTHTSHTHTSHTSHTSHTSHTHTHTSHTHTSHTSHTSHTSHTHTHTHTSHTSHTHTSHTSW
n=3, eps=0.00001: HTSHTHTHTSHTSHTSHTSHTHTSHTSHTSHTHTSHTHTSHTHTHTSHTSHTSHTHTSHTHTSHTHTHTHTHTHTHTSHTHTSHTHTHTSHTSHTHTSHTSHTSHTSHTHTHTHTSHTSHTSHTHTHSSS
n=4, eps=0.01: HTHTHTSHTHTSHTSHTSHTHTSHTSHTHTSHTHTHTSHTSHTSHTSHTHTSHXSSWWWW
n=4, eps=0.001: THTSHTSHTHTSHTSHTSHTSHTSHTHTHTSHTSHTHTSHTHTSHTHTHTHTSHTHTHTSHTHTSHTHTSHTSHTHTHTSHTSHTHXSSS
n=4, eps=0.0001: SHTSHTHTSHTHTHTHTHTSHTHTHTSHTSHTHTHTSHTSHTSHTHTSHTSHTHTHTHTSHTSHTSHTSHTHTSHTSHTSHTSHTHTSHTSHTHTSHTHTSHTSHTHTSWW
n=4, eps=0.00001: SHTHTHTHTHTSHTHTSHTHTSHTSHTHTHTSHTHTHTSHTHTHTHTHTHTHTSHTSHTHTHTSHTHTHTSHTHTSHTSHTHTHTHTSHTSHTHTHTSHTHTHTHTHTSHTSHTHTSHTHTSHTXSWWW
n=5, eps=0.01: SHTSHTHTHTSHTHTHTHTHTSHTHTHTSHTSHTHTSHTHTHTSHT

In [10]:
from utils import density_matrix_fidelity
    
epsilons = sorted(set(eps for _, eps in gridsynth_results.keys()))
print("Unique epsilons:", epsilons)

for n, _ in gridsynth_results.keys():

    theta = np.pi/2**n

    for eps in epsilons:

        gates = gridsynth_results.get((n, eps))
        print(f"  Gates: {gates}")

        @squin.kernel
        def reference():
            q = squin.qalloc(7)
            squin.h(q[0])
            squin.rz(theta, q[0])
            q = steane_code_encoding(q)
            return q
        
        reference_sim = StackMemorySimulator(min_qubits=7)
        reference_task = reference_sim.task(reference)
        reference_result = reference_task.run()
        rho_reference = StackMemorySimulator.reduced_density_matrix(reference_result)

        @squin.kernel
        def synthesized_gate():
            q = squin.qalloc(7)
            q = steane_code_encoding(q)
            q = logical_H(q)
            for gate in gates:
                if gate[0] == "H":
                    q = logical_H(q)
                elif gate[0] == "S":
                    q = logical_S(q)
                elif gate[0] == "T":
                    q = logical_T_teleportation(q)
            return q
        
        synthesized_sim = StackMemorySimulator(min_qubits=7)
        synthesized_task = synthesized_sim.task(synthesized_gate)
        synthesized_result = synthesized_task.run()
        rho_synthesized = StackMemorySimulator.reduced_density_matrix(synthesized_result)

        fidelity = density_matrix_fidelity(rho_reference, rho_synthesized)

        print(f"n={n}, eps={eps}: Fidelity = {fidelity:.6f}")

Unique epsilons: ['0.00001', '0.0001', '0.001', '0.01']
  Gates: HTSHTHTHTSHTSHTSHTSHTHTSHTSHTSHTHTSHTHTSHTHTHTSHTSHTSHTHTSHTHTSHTHTHTHTHTHTHTSHTHTSHTHTHTSHTSHTHTSHTSHTSHTSHTHTHTHTSHTSHTSHTHTHSSS


InterpreterError: qubit allocation exceeds memory, 7 qubits, 8 allocated